# 🔗 Phase 4: Unified Detection Pipeline

**Samsung PRISM - Harmful Content Detection Pipeline**

This notebook combines all detection modules into a single inference pipeline.

## Pipeline Stages
1. 🔫 **Weapon Detection** (YOLOv8)
2. 📝 **Text Classification** (EasyOCR + DistilBERT)
3. 🏷️ **Logo Detection** (YOLOv8)
4. ⚖️ **Decision Engine** → SAFE / UNSAFE

---

**⚠️ Prerequisites:**
- Complete notebooks 01, 02, 03 first
- Trained models saved in Google Drive

**⚠️ Enable GPU in Colab**
- Go to `Runtime` → `Change runtime type` → Select `GPU`

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install ultralytics>=8.0.200 -q
!pip install easyocr>=1.7.1 -q
!pip install transformers>=4.35.0 -q

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import libraries
import os
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from typing import Dict, Any, List, Optional, Union
from dataclasses import dataclass
from enum import Enum
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

## 2️⃣ Load Trained Models

In [ ]:
# Model paths (update these if different)
MODELS_DIR = '/content/drive/MyDrive/samsung_prism/models'

WEAPON_MODEL_PATH = f'{MODELS_DIR}/weapon_detector/best.pt'
LOGO_MODEL_PATH = f'{MODELS_DIR}/logo_detector/best.pt'
TEXT_MODEL_PATH = f'{MODELS_DIR}/text_classifier'

print("Model paths:")
print(f"  Weapon: {WEAPON_MODEL_PATH}")
print(f"  Logo:   {LOGO_MODEL_PATH}")
print(f"  Text:   {TEXT_MODEL_PATH}")

In [ ]:
# Load YOLOv8 models
from ultralytics import YOLO

# Load weapon detector
if os.path.exists(WEAPON_MODEL_PATH):
    weapon_model = YOLO(WEAPON_MODEL_PATH)
    print("✓ Weapon detector loaded")
else:
    print("⚠️ Weapon model not found. Using pretrained YOLOv8.")
    weapon_model = YOLO('yolov8m.pt')

# Load logo detector
if os.path.exists(LOGO_MODEL_PATH):
    logo_model = YOLO(LOGO_MODEL_PATH)
    print("✓ Logo detector loaded")
else:
    print("⚠️ Logo model not found. Using pretrained YOLOv8.")
    logo_model = YOLO('yolov8m.pt')

In [ ]:
# Load NLP model
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

if os.path.exists(TEXT_MODEL_PATH):
    text_tokenizer = DistilBertTokenizer.from_pretrained(TEXT_MODEL_PATH)
    text_model = DistilBertForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)
    text_model.eval()
    if torch.cuda.is_available():
        text_model = text_model.cuda()
    print("✓ Text classifier loaded")
else:
    print("⚠️ Text model not found. Loading pretrained DistilBERT.")
    text_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    text_model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=3
    )
    text_model.eval()
    if torch.cuda.is_available():
        text_model = text_model.cuda()

In [ ]:
# Initialize EasyOCR
import easyocr

print("Initializing EasyOCR...")
ocr_reader = easyocr.Reader(['en'], gpu=True)
print("✓ EasyOCR initialized")

## 3️⃣ Individual Detection Functions

In [ ]:
# Weapon detection function
def detect_weapons(image, model, confidence_threshold=0.5):
    """
    Detect weapons in an image.
    
    Returns:
        Dictionary with detection results
    """
    results = model(image, verbose=False)[0]
    
    detections = []
    for box in results.boxes:
        conf = float(box.conf[0])
        if conf >= confidence_threshold:
            cls_id = int(box.cls[0])
            cls_name = results.names.get(cls_id, f"class_{cls_id}")
            bbox = box.xyxy[0].cpu().numpy().tolist()
            
            detections.append({
                'class': cls_name,
                'confidence': round(conf, 4),
                'bbox': {'x1': int(bbox[0]), 'y1': int(bbox[1]),
                         'x2': int(bbox[2]), 'y2': int(bbox[3])}
            })
    
    return {
        'detected': len(detections) > 0,
        'detections': detections,
        'detection_count': len(detections)
    }

print("✓ Weapon detection function defined")

In [ ]:
# Text detection and classification
LABEL_MAP = {0: 'SAFE', 1: 'PROMOTIONAL', 2: 'ABUSIVE'}

def detect_text(image, ocr_reader, nlp_model, tokenizer, min_ocr_conf=0.3):
    """
    Extract and classify text from image.
    
    Returns:
        Dictionary with OCR and classification results
    """
    # Handle image input
    if isinstance(image, str):
        img_array = cv2.imread(image)
    elif isinstance(image, np.ndarray):
        img_array = image
    else:
        img_array = np.array(image)
    
    # Run OCR
    ocr_results = ocr_reader.readtext(img_array)
    
    extracted_texts = []
    for (bbox, text, confidence) in ocr_results:
        if confidence >= min_ocr_conf:
            extracted_texts.append({
                'text': text,
                'confidence': round(float(confidence), 4)
            })
    
    if not extracted_texts:
        return {
            'text_found': False,
            'combined_text': '',
            'classification': {'label': 'SAFE', 'confidence': 1.0}
        }
    
    # Combine text
    combined_text = ' '.join([t['text'] for t in extracted_texts])
    
    # Classify text
    inputs = tokenizer(combined_text, return_tensors='pt', 
                       truncation=True, max_length=128, padding=True)
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = nlp_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_class = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_class].item()
    
    return {
        'text_found': True,
        'combined_text': combined_text,
        'extracted_texts': extracted_texts,
        'classification': {
            'label': LABEL_MAP[pred_class],
            'confidence': round(confidence, 4)
        }
    }

print("✓ Text detection function defined")

In [ ]:
# Logo detection
COMPETITOR_BRANDS = ['apple', 'google', 'huawei', 'xiaomi', 
                      'oneplus', 'oppo', 'vivo', 'sony', 'lg']

def detect_logos(image, model, confidence_threshold=0.6):
    """
    Detect logos and flag competitors.
    
    Returns:
        Dictionary with logo detection results
    """
    results = model(image, verbose=False)[0]
    
    detections = []
    competitor_detections = []
    
    for box in results.boxes:
        conf = float(box.conf[0])
        if conf >= confidence_threshold:
            cls_id = int(box.cls[0])
            brand = results.names.get(cls_id, f"class_{cls_id}").lower()
            bbox = box.xyxy[0].cpu().numpy().tolist()
            
            detection = {
                'brand': brand,
                'confidence': round(conf, 4),
                'is_competitor': brand in COMPETITOR_BRANDS,
                'bbox': {'x1': int(bbox[0]), 'y1': int(bbox[1]),
                         'x2': int(bbox[2]), 'y2': int(bbox[3])}
            }
            
            detections.append(detection)
            if brand in COMPETITOR_BRANDS:
                competitor_detections.append(detection)
    
    return {
        'detected': len(detections) > 0,
        'competitor_detected': len(competitor_detections) > 0,
        'detections': detections,
        'competitor_detections': competitor_detections
    }

print("✓ Logo detection function defined")

## 4️⃣ Decision Engine

In [ ]:
class SafetyStatus(Enum):
    SAFE = "SAFE"
    UNSAFE = "UNSAFE"


class DecisionEngine:
    """
    Aggregates detection results and makes final SAFE/UNSAFE decision.
    
    Decision Logic:
    - UNSAFE if ANY weapon detected
    - UNSAFE if abusive text detected
    - UNSAFE if promotional text detected
    - UNSAFE if competitor logo detected
    - SAFE otherwise
    """
    
    def __init__(self, weapon_threshold=0.5, logo_threshold=0.6,
                 text_promotional_threshold=0.7, text_abusive_threshold=0.8):
        self.weapon_threshold = weapon_threshold
        self.logo_threshold = logo_threshold
        self.text_promotional_threshold = text_promotional_threshold
        self.text_abusive_threshold = text_abusive_threshold
    
    def evaluate(self, weapon_results, text_results, logo_results):
        """
        Evaluate all detection results.
        
        Returns:
            Dictionary with final decision and all flags
        """
        flags = []
        
        # Check weapons
        if weapon_results.get('detected', False):
            high_conf = [d for d in weapon_results['detections']
                        if d['confidence'] >= self.weapon_threshold]
            if high_conf:
                flags.append({
                    'type': 'WEAPON_DETECTED',
                    'priority': 1,
                    'confidence': max(d['confidence'] for d in high_conf),
                    'objects': [d['class'] for d in high_conf]
                })
        
        # Check text
        if text_results.get('text_found', False):
            classification = text_results.get('classification', {})
            label = classification.get('label', 'SAFE')
            conf = classification.get('confidence', 0)
            
            if label == 'ABUSIVE' and conf >= self.text_abusive_threshold:
                flags.append({
                    'type': 'ABUSIVE_TEXT',
                    'priority': 2,
                    'confidence': conf,
                    'text': text_results.get('combined_text', '')[:100]
                })
            elif label == 'PROMOTIONAL' and conf >= self.text_promotional_threshold:
                flags.append({
                    'type': 'PROMOTIONAL_TEXT',
                    'priority': 4,
                    'confidence': conf,
                    'text': text_results.get('combined_text', '')[:100]
                })
        
        # Check logos
        if logo_results.get('competitor_detected', False):
            competitor_logos = logo_results.get('competitor_detections', [])
            high_conf = [d for d in competitor_logos
                        if d['confidence'] >= self.logo_threshold]
            if high_conf:
                flags.append({
                    'type': 'COMPETITOR_LOGO',
                    'priority': 3,
                    'confidence': max(d['confidence'] for d in high_conf),
                    'brands': [d['brand'] for d in high_conf]
                })
        
        # Sort by priority
        flags.sort(key=lambda x: x['priority'])
        
        # Final decision
        if flags:
            status = SafetyStatus.UNSAFE
            summary = self._build_summary(flags[0])
        else:
            status = SafetyStatus.SAFE
            summary = "No harmful content detected. Content is safe."
        
        return {
            'status': status.value,
            'is_safe': status == SafetyStatus.SAFE,
            'summary': summary,
            'flags': flags,
            'detailed_results': {
                'weapon_detection': weapon_results,
                'text_classification': text_results,
                'logo_detection': logo_results
            }
        }
    
    def _build_summary(self, primary_flag):
        flag_type = primary_flag['type']
        conf = primary_flag['confidence']
        
        if flag_type == 'WEAPON_DETECTED':
            objects = ', '.join(primary_flag['objects'])
            return f"UNSAFE: Weapon detected ({objects}) - Confidence: {conf:.2%}"
        elif flag_type == 'ABUSIVE_TEXT':
            return f"UNSAFE: Abusive text detected - Confidence: {conf:.2%}"
        elif flag_type == 'PROMOTIONAL_TEXT':
            return f"UNSAFE: Promotional content detected - Confidence: {conf:.2%}"
        elif flag_type == 'COMPETITOR_LOGO':
            brands = ', '.join(primary_flag['brands'])
            return f"UNSAFE: Competitor logo detected ({brands}) - Confidence: {conf:.2%}"
        return f"UNSAFE: {flag_type}"


# Initialize decision engine
decision_engine = DecisionEngine()
print("✓ Decision engine initialized")

## 5️⃣ Unified Pipeline

In [ ]:
class UnifiedPipeline:
    """
    Main inference pipeline combining all detectors.
    
    Pipeline Flow:
    1. Input image
    2. Weapon detection
    3. Text detection + classification
    4. Logo detection
    5. Decision engine → SAFE/UNSAFE
    """
    
    def __init__(self, weapon_model, logo_model, text_model, tokenizer, ocr_reader):
        self.weapon_model = weapon_model
        self.logo_model = logo_model
        self.text_model = text_model
        self.tokenizer = tokenizer
        self.ocr_reader = ocr_reader
        self.decision_engine = DecisionEngine()
        
        print("✓ Unified pipeline initialized")
    
    def process_image(self, image):
        """
        Process a single image through all detection stages.
        
        Args:
            image: Image path or numpy array
            
        Returns:
            Complete detection results with final verdict
        """
        # Load image
        if isinstance(image, str):
            image_path = image
            img_array = cv2.imread(image)
        else:
            image_path = None
            img_array = image
        
        # Stage 1: Weapon Detection
        weapon_results = detect_weapons(img_array, self.weapon_model)
        
        # Stage 2: Text Detection + Classification
        text_results = detect_text(img_array, self.ocr_reader, 
                                   self.text_model, self.tokenizer)
        
        # Stage 3: Logo Detection
        logo_results = detect_logos(img_array, self.logo_model)
        
        # Stage 4: Decision Engine
        decision = self.decision_engine.evaluate(
            weapon_results, text_results, logo_results
        )
        
        return {
            'input': {'type': 'image', 'path': image_path},
            **decision
        }
    
    def process_video(self, video_path, frame_interval=30, max_frames=50):
        """
        Process video by analyzing frames.
        
        Args:
            video_path: Path to video file
            frame_interval: Analyze every Nth frame
            max_frames: Maximum frames to analyze
            
        Returns:
            Aggregated results for video
        """
        cap = cv2.VideoCapture(video_path)
        
        frame_count = 0
        analyzed_count = 0
        unsafe_frames = []
        frame_results = []
        
        while cap.isOpened() and analyzed_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % frame_interval == 0:
                result = self.process_image(frame)
                result['frame_index'] = analyzed_count
                frame_results.append(result)
                
                if not result['is_safe']:
                    unsafe_frames.append({
                        'frame_index': analyzed_count,
                        'flags': result['flags']
                    })
                
                analyzed_count += 1
            
            frame_count += 1
        
        cap.release()
        
        # Aggregate results
        is_safe = len(unsafe_frames) == 0
        
        if is_safe:
            summary = f"Video analyzed ({analyzed_count} frames). No harmful content detected."
        else:
            summary = f"Video UNSAFE: {len(unsafe_frames)}/{analyzed_count} frames contain harmful content."
        
        return {
            'input': {'type': 'video', 'path': video_path},
            'status': 'SAFE' if is_safe else 'UNSAFE',
            'is_safe': is_safe,
            'summary': summary,
            'analysis': {
                'total_frames_analyzed': analyzed_count,
                'unsafe_frames_count': len(unsafe_frames)
            },
            'unsafe_frames': unsafe_frames
        }


# Initialize pipeline
pipeline = UnifiedPipeline(
    weapon_model=weapon_model,
    logo_model=logo_model,
    text_model=text_model,
    tokenizer=text_tokenizer,
    ocr_reader=ocr_reader
)

## 6️⃣ Demo: Process Sample Images

In [ ]:
# Create test images for demo
from PIL import Image, ImageDraw, ImageFont

# Test 1: Safe image
img1 = Image.new('RGB', (400, 300), color='white')
draw1 = ImageDraw.Draw(img1)
draw1.text((50, 100), "Welcome to our store", fill='black')
img1.save('/content/test_safe.jpg')

# Test 2: Promotional image
img2 = Image.new('RGB', (400, 300), color='yellow')
draw2 = ImageDraw.Draw(img2)
draw2.text((50, 50), "BUY NOW!", fill='red')
draw2.text((50, 100), "50% OFF - Limited Time!", fill='blue')
draw2.text((50, 150), "Sale ends today!", fill='black')
img2.save('/content/test_promotional.jpg')

# Test 3: Competitor logo (simulated)
img3 = Image.new('RGB', (400, 300), color='white')
draw3 = ImageDraw.Draw(img3)
draw3.rectangle([100, 50, 300, 200], outline='black', width=2)
draw3.text((150, 100), "APPLE", fill='black')
img3.save('/content/test_logo.jpg')

print("✓ Test images created")

In [ ]:
# Process test images
test_images = [
    ('/content/test_safe.jpg', 'Safe Content'),
    ('/content/test_promotional.jpg', 'Promotional Content'),
    ('/content/test_logo.jpg', 'Competitor Logo')
]

print("\n" + "="*60)
print("🔍 UNIFIED PIPELINE - DEMO RESULTS")
print("="*60)

for image_path, description in test_images:
    result = pipeline.process_image(image_path)
    
    status_emoji = "✅" if result['is_safe'] else "🚨"
    
    print(f"\n{'-'*60}")
    print(f"📌 {description}")
    print(f"{status_emoji} Status: {result['status']}")
    print(f"📝 Summary: {result['summary']}")
    
    if result['flags']:
        print(f"⚠️  Flags:")
        for flag in result['flags']:
            print(f"    - {flag['type']} (conf: {flag['confidence']:.2%})")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (image_path, description) in zip(axes, test_images):
    img = Image.open(image_path)
    result = pipeline.process_image(image_path)
    
    ax.imshow(img)
    
    # Set title with status
    color = 'green' if result['is_safe'] else 'red'
    status_emoji = "✅ SAFE" if result['is_safe'] else "🚨 UNSAFE"
    ax.set_title(f"{description}\n{status_emoji}", fontsize=12, color=color)
    ax.axis('off')

plt.suptitle('Unified Pipeline Detection Results', fontsize=14)
plt.tight_layout()
plt.savefig('/content/pipeline_results.png', dpi=150)
plt.show()

## 7️⃣ JSON Output Format

In [ ]:
# Complete example output
example_output = {
    "status": "UNSAFE",
    "is_safe": False,
    "summary": "UNSAFE: Weapon detected (gun) - Confidence: 92.50%",
    "flags": [
        {
            "type": "WEAPON_DETECTED",
            "priority": 1,
            "confidence": 0.925,
            "objects": ["gun"]
        },
        {
            "type": "PROMOTIONAL_TEXT",
            "priority": 4,
            "confidence": 0.85,
            "text": "BUY NOW! 50% OFF!"
        }
    ],
    "detailed_results": {
        "weapon_detection": {
            "detected": True,
            "detections": [
                {"class": "gun", "confidence": 0.925, 
                 "bbox": {"x1": 100, "y1": 150, "x2": 300, "y2": 400}}
            ]
        },
        "text_classification": {
            "text_found": True,
            "combined_text": "BUY NOW! 50% OFF!",
            "classification": {"label": "PROMOTIONAL", "confidence": 0.85}
        },
        "logo_detection": {
            "detected": False,
            "competitor_detected": False
        }
    }
}

print("\n" + "="*60)
print("📋 EXAMPLE JSON OUTPUT")
print("="*60)
print(json.dumps(example_output, indent=2))

## 8️⃣ Process Your Own Images

In [ ]:
# Upload and process your own image
from google.colab import files

print("Upload an image to analyze:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Save uploaded file
    image_path = f'/content/{filename}'
    
    # Process
    result = pipeline.process_image(image_path)
    
    # Display results
    print("\n" + "="*60)
    print(f"📌 Analysis for: {filename}")
    print("="*60)
    
    status_emoji = "✅" if result['is_safe'] else "🚨"
    print(f"\n{status_emoji} STATUS: {result['status']}")
    print(f"\n📝 Summary: {result['summary']}")
    
    if result['flags']:
        print(f"\n⚠️  Issues Detected:")
        for i, flag in enumerate(result['flags'], 1):
            print(f"   {i}. {flag['type']}")
            print(f"      Confidence: {flag['confidence']:.2%}")
    
    # Show image
    img = Image.open(image_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    color = 'green' if result['is_safe'] else 'red'
    plt.title(f"Result: {result['status']}", fontsize=16, color=color)
    plt.axis('off')
    plt.show()

## 9️⃣ Save Pipeline for Production

In [ ]:
# Save complete pipeline configuration
pipeline_config = {
    "model_paths": {
        "weapon_detector": WEAPON_MODEL_PATH,
        "logo_detector": LOGO_MODEL_PATH,
        "text_classifier": TEXT_MODEL_PATH
    },
    "thresholds": {
        "weapon_confidence": 0.5,
        "logo_confidence": 0.6,
        "text_promotional": 0.7,
        "text_abusive": 0.8
    },
    "competitor_brands": COMPETITOR_BRANDS,
    "text_labels": {"0": "SAFE", "1": "PROMOTIONAL", "2": "ABUSIVE"}
}

# Save config
config_path = '/content/drive/MyDrive/samsung_prism/pipeline_config.json'
with open(config_path, 'w') as f:
    json.dump(pipeline_config, f, indent=2)

print(f"\n✓ Pipeline configuration saved to: {config_path}")

In [ ]:
print("\n" + "="*60)
print("🎉 UNIFIED PIPELINE COMPLETE!")
print("="*60)
print("\n📦 Components:")
print("   1. Weapon Detection (YOLOv8)")
print("   2. Text Classification (EasyOCR + DistilBERT)")
print("   3. Logo Detection (YOLOv8)")
print("   4. Decision Engine (SAFE/UNSAFE)")
print("\n📁 Saved Models:")
print(f"   - Weapon: {WEAPON_MODEL_PATH}")
print(f"   - Logo: {LOGO_MODEL_PATH}")
print(f"   - Text: {TEXT_MODEL_PATH}")
print(f"   - Config: {config_path}")
print("\n✅ Ready for deployment!")

---

## Summary

| Component | Model | Output |
|-----------|-------|--------|
| Weapon Detection | YOLOv8m | Bounding boxes + class |
| Text Classification | DistilBERT | SAFE/PROMOTIONAL/ABUSIVE |
| Logo Detection | YOLOv8m | Brand name + competitor flag |
| **Final Output** | Decision Engine | **SAFE / UNSAFE** |

### Decision Logic
- **UNSAFE** if ANY of: weapon detected, abusive text, promotional text, competitor logo
- **SAFE** otherwise

### Pipeline Config
Saved to: `/content/drive/MyDrive/samsung_prism/pipeline_config.json`